# π0 模型推理验证

修改 **参数配置区** 即可切换不同权重/任务，按顺序执行所有 cell 即可完成推理验证。

内存需求：权重大小约 7 GB，加载时需约 2× 权重大小的可用内存（≈ 15 GB）。

In [ ]:
from pathlib import Path

# ==== 参数配置区 ====
WEIGHT_DIR = "checkpoints/pi05_spiritai_lora_pytorch"
CONFIG_NAME = "pi05_spiritai_lora"
TASK_PROMPT = "Assemble the cardboard box by erecting the flat sheet and folding the side flaps"

# "fake": 使用 make_spiritai_example() 生成随机数据
# 或直接赋值一个观测 dict (需包含 cam_high/cam_left_wrist/cam_right_wrist/各state键/prompt)
DATA_MODE = "fake"
CUSTOM_OBS = None  # DATA_MODE != "fake" 时使用

BENCHMARK_ITERS = 5  # warmup 后压测次数，0 跳过

In [ ]:
import numpy as np
import torch
from openpi.models import model as _model
from openpi.policies import spiritai_policy
from openpi.policies import policy_config as _policy_config
from openpi.training import config as _config

# Resolve weight path relative to repo root (works regardless of cwd)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / "pyproject.toml").exists():
    _repo_root = _repo_root.parent
weight_path = (_repo_root / WEIGHT_DIR).resolve()
if not weight_path.exists():
    raise FileNotFoundError(f"WEIGHT_DIR not found: {weight_path}")
if not (weight_path / "model.safetensors").exists():
    raise FileNotFoundError(f"model.safetensors not found in {weight_path}")

# Memory check
weight_bytes = (weight_path / "model.safetensors").stat().st_size
import psutil
avail_gb = psutil.virtual_memory().available / 1e9
need_gb = weight_bytes * 2 / 1e9  # rough: need ~2x weight size for loading
print(f"Repo root   : {_repo_root}")
print(f"Weights     : {weight_path} ({weight_bytes / 1e9:.1f} GB)")
print(f"Available RAM: {avail_gb:.1f} GB (need ~{need_gb:.0f} GB for loading)")
if avail_gb < need_gb:
    print(f"\n[WARN] 内存可能不足！可用 {avail_gb:.1f} GB < 预估需要 {need_gb:.0f} GB。加载时可能 OOM 崩溃。")

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU VRAM    : {vram_gb:.1f} GB ({torch.cuda.get_device_name(0)})")
else:
    print("GPU VRAM    : N/A (will use CPU, inference slow)")

## 构建观测数据

In [ ]:
if DATA_MODE == "fake":
    observation = spiritai_policy.make_spiritai_example()
    observation["prompt"] = TASK_PROMPT
else:
    observation = dict(CUSTOM_OBS)
    observation.setdefault("prompt", TASK_PROMPT)

state_dim = sum(np.asarray(observation[k]).flatten().shape[0] for k in spiritai_policy.STATE_KEYS)
print(f"state dim: {state_dim} | prompt: {observation['prompt']}")
for k in ("cam_high", "cam_left_wrist", "cam_right_wrist"):
    print(f"  {k}: {observation[k].shape} {observation[k].dtype}")

## 加载权重并推理

In [ ]:
import time

config = _config.get_config(CONFIG_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Config: {config.name} | pi05={config.model.pi05} | action_dim={config.model.action_dim} | action_horizon={config.model.action_horizon}")
print(f"Device: {device}")

try:
    t0 = time.time()
    policy = _policy_config.create_trained_policy(config, str(weight_path), pytorch_device=device)
    print(f"Policy loaded ({time.time() - t0:.1f}s) | pytorch={policy._is_pytorch_model}")
except RuntimeError as e:
    if "out of memory" in str(e).lower() or "CUDA" in str(e):
        print(f"[ERROR] CUDA OOM: {e}")
        print("尝试回退到 CPU ...")
        torch.cuda.empty_cache()
        device = "cpu"
        t0 = time.time()
        policy = _policy_config.create_trained_policy(config, str(weight_path), pytorch_device=device)
        print(f"Policy loaded on CPU ({time.time() - t0:.1f}s)")
    else:
        raise

t0 = time.time()
result = policy.infer(observation)
print(f"Inference: {((time.time() - t0) * 1000):.0f} ms")

actions = result["actions"]
expected = (config.model.action_horizon, spiritai_policy.ACTION_DIM)
assert actions.shape == expected, f"shape {actions.shape} != expected {expected}"
print(f"Actions shape: {actions.shape} | dtype: {actions.dtype} | range: [{actions.min():.4f}, {actions.max():.4f}]")

if BENCHMARK_ITERS > 0:
    for _ in range(3):
        policy.infer(observation)
    times = []
    for _ in range(BENCHMARK_ITERS):
        t0 = time.time(); policy.infer(observation); times.append((time.time() - t0) * 1000)
    print(f"Benchmark ({BENCHMARK_ITERS} iters): mean={np.mean(times):.0f}ms std={np.std(times):.0f}ms")

del policy
print("Done.")